# Contrastive Language Models: CLM은 어떻게 학습되는가

이 노트북은 **Contrastive Language Models: A System One Model for Fast and Generalizable Decision-Making**의 공개 설명과 학습 코드를 따라가며 CLM의 학습 과정을 작은 PyTorch 실험으로 재현합니다.

## 먼저 구분할 것

- CLM은 2026-09-23 공개된 technical report이며 현재 peer-reviewed 논문은 아닙니다.
- 공개 CLM-8B는 **Qwen3-8B의 last-token pooled embedding을 고정**하고, state/action용 projection head 두 개와 logit scale을 학습합니다.
- 이 노트북의 작은 합성 embedding은 학습 수학과 코드 경로를 확인하기 위한 것입니다. 60M/30M/1M 규모의 보고 결과를 재현하지 않습니다.
- 저자 보고 성능과 Jev 비교는 독립 재현 결과가 아닙니다.

### 출처

- [Technical report](https://contrastive-lm.notion.site/)
- [공개 저장소](https://github.com/Contrastive-LM/CLM)
- [CLM-v0.1-8B model card](https://huggingface.co/Contrastive-LM/CLM-v0.1-8B)
- [사용자 Notion 정리](https://app.notion.com/p/3e963f83d98981e0a0f3f55dca109477)


## 전체 학습 흐름

```text
state text ── frozen Qwen3-8B ── 4096d embedding ── state projection head  ── L2 normalize ─┐
                                                                                           ├─ cosine × exp(logit_scale)
action text ─ frozen Qwen3-8B ── 4096d embedding ── action projection head ── L2 normalize ─┘

Stage 1: 약 60M Nemotron Q&A                 → broad state-action alignment
Stage 2: 약 30M synthetic hard negatives     → plausible-but-wrong action 구분
Stage 3: 약 1M agentic trajectory transitions → 실제 agent state/action에 적응
         + post-training mixture의 40% Q&A replay
```

추론에서는 autoregressive token을 생성하지 않습니다. 미리 계산한 action embedding을 재사용하고 새 state와 각 action의 compatibility를 계산합니다.


In [ ]:
from pathlib import Path
import copy, importlib.util, inspect, json, math, os, subprocess, sys, tempfile
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 7
torch.manual_seed(SEED)
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "device", DEVICE)


## 1. 공개 학습 코드를 고정된 commit에서 읽기

노트북은 `bb42c6c`의 `train/finetune.py`를 import합니다. 로컬 경로는 `CLM_REPO`로 지정할 수 있습니다. 없으면 임시 디렉터리에 공개 저장소를 내려받습니다.


In [ ]:
REV = "bb42c6c5bf914fd449bed2f6ca65be80602cb1f7"
locations = [Path(os.environ["CLM_REPO"])] if os.getenv("CLM_REPO") else []
locations += [Path.cwd().parent / "CLM", Path(tempfile.gettempdir()) / "learning-jev-clm"]
repo = next((p.resolve() for p in locations if (p / "train/finetune.py").is_file()), None)
if repo is None:
    repo = locations[-1]
    subprocess.run(["git", "clone", "https://github.com/Contrastive-LM/CLM.git", str(repo)], check=True)
current = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
if current != REV and repo == locations[-1]:
    subprocess.run(["git", "-C", str(repo), "checkout", "--detach", REV], check=True)
elif current != REV:
    print(f"주의: 로컬 commit {current[:12]} != 분석 기준 {REV[:12]}")

spec = importlib.util.spec_from_file_location("clm_finetune", repo / "train/finetune.py")
clm_train = importlib.util.module_from_spec(spec)
spec.loader.exec_module(clm_train)
print("CLM source:", repo, "commit:", current[:12])


## 2. 공개 코드의 핵심: 양방향 in-batch InfoNCE

batch에 정답 pair `(sᵢ, aᵢ)`가 B개 있으면 모든 state와 action의 `B × B` score matrix를 만듭니다.

\[
L = rac{1}{2}\left[CE(SA^T, [0,\dots,B-1]) + CE(AS^T,[0,\dots,B-1])ight]
\]

대각선은 positive pair, 나머지는 in-batch negative입니다. 공개 fine-tuning 코드는 같은 `(task_id, step_idx)` group의 다른 항목을 false negative로 쓰지 않도록 mask합니다. `logit_scale = log(1/τ)`도 학습 대상이며 `exp(logit_scale)`은 최대 100으로 제한됩니다.


In [ ]:
print(inspect.getsource(clm_train._clm_loss))

## 3. 작은 frozen encoder 환경 만들기

실제 학습은 Qwen3-8B embedding을 먼저 저장하고 backbone을 고정합니다. 여기서는 같은 잠재 의미가 state와 action에서 서로 다른 frozen embedding으로 관측되는 합성 환경을 만듭니다. 두 projection head가 두 공간을 다시 맞추는지 확인합니다.


In [ ]:
LATENT, HIDDEN, PROJ = 12, 32, 12
state_encoder = torch.randn(LATENT, HIDDEN, device=DEVICE)
action_encoder = torch.randn(LATENT, HIDDEN, device=DEVICE)

def frozen_embed(z, encoder):
    noise = 0.05 * torch.randn(z.shape[0], HIDDEN, device=DEVICE)
    return F.normalize(z @ encoder + noise, dim=-1)

def paired_embeddings(n, state_map=state_encoder, action_map=action_encoder, shift=0.0):
    latent = F.normalize(torch.randn(n, LATENT, device=DEVICE) + shift, dim=-1)
    return frozen_embed(latent, state_map), frozen_embed(latent, action_map), latent

pre_states, pre_actions, pre_latent = paired_embeddings(256)
print(pre_states.shape, pre_actions.shape, "requires_grad:", pre_states.requires_grad)


## 4. Stage 1 — broad state-action alignment

저자 설명에서는 약 60M Nemotron Q&A pair를 사용합니다. 질문이 state, 정답이 action입니다. 아래 실험은 공개 `make_head`와 `_clm_loss`를 그대로 사용합니다. 실제 공개 기본 head는 4096차원 입력, MLP width 1536, depth 3, projection 512이지만 toy에서는 작게 줄입니다.


In [ ]:
state_head = clm_train.make_head(width=48, depth=3, proj=PROJ, hidden=HIDDEN, layernorm=True).to(DEVICE)
action_head = clm_train.make_head(width=48, depth=3, proj=PROJ, hidden=HIDDEN, layernorm=True).to(DEVICE)
logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07), device=DEVICE))
params = list(state_head.parameters()) + list(action_head.parameters()) + [logit_scale]
optimizer = torch.optim.AdamW(params, lr=3e-3)

def retrieval_accuracy(states, actions):
    with torch.no_grad():
        zs = F.normalize(state_head(states), dim=-1)
        za = F.normalize(action_head(actions), dim=-1)
        return (zs @ za.T).argmax(1).eq(torch.arange(len(states), device=DEVICE)).float().mean().item()

print("before:", retrieval_accuracy(pre_states, pre_actions))
for step in range(120):
    idx = torch.randint(0, len(pre_states), (64,), device=DEVICE)
    # 모든 pair가 서로 다른 group인 toy batch
    codes = torch.arange(len(idx), device=DEVICE)
    loss, similarity, labels = clm_train._clm_loss(
        state_head, action_head, logit_scale, pre_states[idx], pre_actions[idx], codes
    )
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    optimizer.step()

print({"loss": round(loss.item(), 4), "retrieval_top1": round(retrieval_accuracy(pre_states, pre_actions), 4),
       "learned_temperature": round(float(logit_scale.detach().exp().reciprocal()), 4),
       "score_matrix": tuple(similarity.shape)})


## 5. Stage 2 — synthetic hard-negative mid-training

random in-batch negative는 쉽게 구분될 수 있습니다. 저자 recipe는 약 30M개의 **의미상 그럴듯하지만 틀린 답변**을 Gemini 2.5 Flash-Lite로 생성해 refinement 단계에서 추가합니다.

공개 report의 state→action hard-negative 항은 다음과 같습니다.

\[
-\lograc{e^{s_i^Ta_i/τ}}{e^{s_i^Ta_i/τ}+\sum_k e^{s_i^Th_{ik}/τ}}
\]

아래 `hard_negative_loss`는 이 식을 작은 데이터에 옮긴 교육용 구현입니다. 대규모 pre/mid-training 전체 코드는 현재 공개 fine-tuning branch에 포함되어 있지 않으므로, 이 셀을 공개 pipeline의 완전한 재현으로 해석하면 안 됩니다.


In [ ]:
# positive와 매우 비슷하지만 일관된 오답 신호를 가진 action embedding
hard_actions = F.normalize(pre_actions + 0.10 * torch.ones_like(pre_actions), dim=-1)

def hard_negative_loss(states, positives, negatives):
    zs = F.normalize(state_head(states), dim=-1)
    zp = F.normalize(action_head(positives), dim=-1)
    zn = F.normalize(action_head(negatives), dim=-1)
    scale = logit_scale.exp().clamp(max=100.0)
    logits = scale * torch.stack([(zs * zp).sum(-1), (zs * zn).sum(-1)], dim=1)
    return F.cross_entropy(logits, torch.zeros(len(states), dtype=torch.long, device=DEVICE))

def hard_margin():
    with torch.no_grad():
        zs = F.normalize(state_head(pre_states), dim=-1)
        pos = (zs * F.normalize(action_head(pre_actions), dim=-1)).sum(-1)
        neg = (zs * F.normalize(action_head(hard_actions), dim=-1)).sum(-1)
        return float((pos - neg).mean())

before_margin = hard_margin()
for step in range(80):
    idx = torch.randint(0, len(pre_states), (64,), device=DEVICE)
    codes = torch.arange(len(idx), device=DEVICE)
    base, _, _ = clm_train._clm_loss(state_head, action_head, logit_scale,
                                      pre_states[idx], pre_actions[idx], codes)
    loss = base + hard_negative_loss(pre_states[idx], pre_actions[idx], hard_actions[idx])
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    optimizer.step()
print({"hard_margin_before": round(before_margin, 4), "hard_margin_after": round(hard_margin(), 4)})


## 6. Stage 3 — agentic post-training + replay

저자 설명은 약 1M agent trajectory의 각 step을 `(현재 context, 실제 선택 action)` pair로 바꿉니다. post-training mixture는 agentic pair 60%, Nemotron Q&A replay 40%입니다. replay는 기존의 일반적인 구분 능력이 무너지는 것을 줄입니다.

아래에서는 agent domain의 embedding 분포를 의도적으로 바꾸어 adaptation을 관찰합니다. 합성 변환은 Qwen 동작을 모델링하지 않으며 distribution shift를 눈에 보이게 만들기 위한 장치입니다.


In [ ]:
agent_state_map = torch.randn_like(state_encoder)
agent_action_map = torch.randn_like(action_encoder)
agent_states, agent_actions, _ = paired_embeddings(160, agent_state_map, agent_action_map, shift=0.3)

before_agent = retrieval_accuracy(agent_states, agent_actions)
before_replay = retrieval_accuracy(pre_states, pre_actions)
for step in range(100):
    # 60% agentic + 40% Q&A replay
    ia = torch.randint(0, len(agent_states), (38,), device=DEVICE)
    ip = torch.randint(0, len(pre_states), (26,), device=DEVICE)
    states = torch.cat([agent_states[ia], pre_states[ip]])
    actions = torch.cat([agent_actions[ia], pre_actions[ip]])
    codes = torch.arange(len(states), device=DEVICE)
    loss, _, _ = clm_train._clm_loss(state_head, action_head, logit_scale, states, actions, codes)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    optimizer.step()

print({"agent_top1_before": round(before_agent, 4),
       "agent_top1_after": round(retrieval_accuracy(agent_states, agent_actions), 4),
       "pretrain_top1_before_posttrain": round(before_replay, 4),
       "pretrain_top1_after_replay": round(retrieval_accuracy(pre_states, pre_actions), 4)})


## 7. 추론: typed decision도 같은 scoring 문제

`Choice`는 후보 action, `Noul`은 true/false action, `Score`는 ordered level action으로 바뀝니다. 각 후보 점수에 softmax를 적용합니다. Score는 level index의 기댓값을 반환합니다.


In [ ]:
with torch.no_grad():
    one_state = agent_states[:1]
    candidates = agent_actions[:4]
    zs = F.normalize(state_head(one_state), dim=-1)
    za = F.normalize(action_head(candidates), dim=-1)
    logits = logit_scale.exp().clamp(max=100) * (zs @ za.T).squeeze(0)
    probabilities = logits.softmax(-1)
    expected_level = sum(i * float(p) for i, p in enumerate(probabilities))
print({"logits": [round(float(x), 3) for x in logits],
       "probabilities": [round(float(x), 4) for x in probabilities],
       "choice": int(probabilities.argmax()), "score_expectation": round(expected_level, 4)})


## NanoJev와 학습 경로 비교

| 항목 | NanoJev | CLM |
| --- | --- | --- |
| encoder 입력 | `state + question + candidate` path | state와 action을 각각 encode |
| 학습 대상 | backbone + scalar/set decision head | 공개 fine-tuning 기준 frozen backbone + state/action projection heads + temperature |
| 기본 objective | complete-question CE | bidirectional InfoNCE |
| candidate 재사용 | 현재 공개 경로는 prefix 반복 | action embedding과 projection을 cache 가능 |
| 후보 상호작용 | Choice set attention | 공통 후보 pool에서 dot product와 softmax |
| 확률 해석 | candidate set 상대 분포 | candidate set 상대 분포; absolute calibration은 별도 검증 필요 |

CLM이 Jev의 내부 구조와 같다는 증거는 없습니다. CLM은 System One 결정을 dual-encoder retrieval 문제로 구성한 공개 대안입니다.


## 실제 데이터로 다음에 해볼 실험

1. agent trajectory를 `state`, `action`, `task_id`, `step_idx`로 변환합니다.
2. action 후보에 쉬운 random negative와 같은 tool family의 hard negative를 각각 넣습니다.
3. **task 단위**로 train/validation/test를 분리해 같은 task의 step이 양쪽에 새지 않게 합니다.
4. ranking top-1, mean rank뿐 아니라 NLL/Brier/ECE와 `none-of-the-above` 또는 abstain을 따로 평가합니다.
5. CQRouter라면 action ranking 외에 `Q_success(s,a)`와 `Q_cost(s,a)` head를 비교하고 `Q_success - λQ_cost`로 route합니다.

공개 fine-tuning 진입점 예시:

```bash
python train/finetune.py \
  --task clm \
  --data transitions.jsonl \
  --init-ckpt CLM_v0.1-8B.pt \
  --out-dir runs/my-agent
```

Qwen3-8B embedding 생성에는 vLLM과 충분한 GPU 메모리가 필요합니다. 이 노트북의 toy 부분은 Apple Silicon/CPU에서도 실행됩니다.
